## STEP 1 — Feature Audit and Final Feature Strategy

Before splitting the data or training any machine learning model, we need to conduct a structured audit of the provisional modeling features.

The purpose of this step is to determine:

* which columns are appropriate for modeling;
* which columns are redundant representations of the same information;
* which features are numeric or categorical;
* which categorical features have high cardinality;
* which features are extremely sparse;
* whether any variables may introduce target leakage;
* which transformations should be handled later through the preprocessing pipeline.

At this stage, features will **not automatically be dropped solely because they contain missing values, have high cardinality, or appear sparse**. Each feature will first be reviewed in the context of:

1. predictive usefulness;
2. clinical and operational realism;
3. potential data leakage;
4. redundancy with engineered features;
5. interpretability;
6. preprocessing complexity; and
7. expected availability at prediction time.

The current modeling dataset contains 67 columns. Eight columns are already excluded from the provisional feature set because they are identifiers, the target, the original target source, the eligibility flag used to construct the modeling population, or redundant numerical identifiers that have already been mapped to descriptive categories.

This leaves **59 provisional model features**.

The feature audit will classify the provisional features into the following categories:

* Numeric features
* Low-cardinality categorical features
* High-cardinality categorical features
* Sparse features
* Potentially redundant features
* Potentially leaking features
* Features requiring special transformation or preprocessing

No preprocessing will be fitted during this stage. The purpose is to understand the feature space and define a justified final modeling strategy before creating the train/validation/test split.


In [5]:
#load the dataset
import pandas as pd

df_clean_feature = pd.read_csv("../Data/Processed/diabetes_cleaned_data.csv")
df_model_feature = pd.read_csv('../Data/Processed/diabetes_modeling_data.csv')


print("df_clean_feature:", df_clean_feature.shape)
print("df_model_feature:", df_model_feature.shape)

print("\nColumns in df_model_feature:")
print(df_model_feature.columns.tolist())


df_clean_feature: (101766, 67)
df_model_feature: (100114, 67)

Columns in df_model_feature:
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted', 'readmitted_30d', 'admission_type', 'discharge_disposition', 'admission_source', 'diag_1_group', 'diag_2_group', 'di

# PHASE 4 — MACHINE LEARNING MODEL DEVELOPMENT

## Dataset Loading and Modeling Dataset Confirmation

This notebook begins the machine learning development phase of the hospital readmission prediction project.

The data cleaning, feature engineering, data integrity validation, and modeling population construction have already been completed in the previous project phase. To maintain a reproducible workflow, the resulting datasets were saved and reloaded for machine learning development.

Two datasets are available:

### Cleaned Feature Dataset

`df_clean_feature` contains the complete cleaned and feature-engineered dataset.

**Shape:** 101,766 encounters × 67 columns

### Modeling Dataset

`df_model_feature` contains the final modeling population.

**Shape:** 100,114 encounters × 67 columns

Death-related encounters were removed from the modeling population because 30-day hospital readmission is not a meaningful prediction outcome for encounters where the patient was discharged to a death-related disposition.

A total of **1,652 death-related encounters** were removed.

The modeling dataset will be used for all subsequent machine learning development steps.

The target variable is:

`readmitted_30d`

Where:

* `1` = patient was readmitted within 30 days
* `0` = patient was not readmitted within 30 days

Before training any machine learning model, the next step is to perform a structured feature audit. The objective is to review all provisional features, identify redundancy, understand feature types and cardinality, evaluate sparsity, and identify any features that require special preprocessing or exclusion.

No model training or preprocessing will be performed until the feature strategy has been formally defined.


## STEP 1.1 — Initial Feature Audit

Before defining the final feature strategy, an initial audit will be performed on every column in the modeling dataset.

The purpose of this audit is to understand the structure and characteristics of the available variables before making any feature selection decisions.

For each column, the following information will be reviewed:

* data type;
* number of missing values;
* percentage of missing values;
* number of unique values;
* percentage of unique values;
* representative sample values.

The initial audit is intended to provide evidence for later classification of features into categories such as:

* identifiers and excluded columns;
* target-related columns;
* numeric features;
* low-cardinality categorical features;
* high-cardinality categorical features;
* sparse features;
* potentially redundant features;
* features requiring special preprocessing.

At this stage, no feature will be automatically removed based only on missingness, cardinality, or sparsity. Final feature decisions will be made after reviewing the clinical, operational, and machine learning implications of each variable.

The modeling dataset contains **100,114 hospital encounters and 67 columns**. Eight columns have already been identified as excluded from the provisional feature set, leaving **59 provisional model features** for detailed review.


In [6]:
# ============================================================
# STEP 1.1 — INITIAL FEATURE AUDIT
# ============================================================

feature_audit = pd.DataFrame({
    "feature": df_model_feature.columns,
    "data_type": df_model_feature.dtypes.astype(str).values,
    "missing_count": df_model_feature.isna().sum().values,
    "missing_pct": (
        df_model_feature.isna().mean().mul(100).round(2).values
    ),
    "unique_count": df_model_feature.nunique(dropna=True).values,
    "unique_pct": (
        (df_model_feature.nunique(dropna=True) / len(df_model_feature))
        .mul(100)
        .round(2)
        .values
    ),
    "sample_values": [
        df_model_feature[col]
        .dropna()
        .astype(str)
        .unique()[:5]
        .tolist()
        for col in df_model_feature.columns
    ]
})

# Sort columns alphabetically for easier review
feature_audit = feature_audit.sort_values(
    by="feature"
).reset_index(drop=True)

print(f"Total columns audited: {len(feature_audit)}")

feature_audit

Total columns audited: 67


,feature,data_type,missing_count,missing_pct,unique_count,unique_pct,sample_values
0,A1C_documented,int64,0,0.00,2,0.00,"[0, 1]"
1,A1C_level,float64,83238,83.14,3,0.00,"[1.0, 2.0, 0.0]"
2,A1Cresult,str,0,0.00,4,0.00,"[Not_Documented, >7, >8, Norm]"
3,acarbose,str,0,0.00,4,0.00,"[No, Steady, Up, Down]"
4,acetohexamide,str,0,0.00,2,0.00,"[No, Steady]"
...,...,...,...,...,...,...,...
62,tolazamide,str,0,0.00,3,0.00,"[No, Steady, Up]"
63,tolbutamide,str,0,0.00,2,0.00,"[No, Steady]"
64,troglitazone,str,0,0.00,2,0.00,"[No, Steady]"
65,weight,str,96958,96.85,9,0.01,"[[75-100), [50-75), [0-25), [100-125), [25-50)]"


## STEP 1.2 — Initial Feature Classification

The initial feature audit provides the technical characteristics of every column, including data type, missingness, cardinality, and representative values. However, machine learning feature strategy should be based on the meaning of each variable rather than its storage data type alone.

For example, an integer column may represent either a genuine numerical quantity or a categorical code. Similarly, a floating-point column may represent a continuous measurement or an encoded ordinal category.

Therefore, the next stage of the feature audit is to classify each variable according to its analytical and modeling role.

The dataset contains the following broad feature categories:

### Identifiers

Encounter and patient identifiers are retained in the dataset for validation and splitting purposes but are not intended to be directly used as predictive features.

### Target-related variables

The original `readmitted` column is excluded because it was used to construct the binary target. The engineered `readmitted_30d` column is the target variable.

### Modeling population construction variables

The `death_related_disposition` variable was used to construct the final modeling population. Because this variable defines eligibility rather than an independent predictor available for the intended prediction task, it is excluded from model features.

### Redundant numerical category codes

The numerical identifiers for admission type, discharge disposition, and admission source have corresponding descriptive mapped versions. The descriptive variables are initially preferred because they provide the same conceptual information while improving interpretability.

### Numerical utilization and clinical features

Variables representing counts, procedures, hospital utilization, medications, and diagnostic burden are treated as genuine numerical features.

### Categorical demographic and clinical features

Variables such as race, gender, age group, payer information, medical specialty, and grouped diagnoses represent categorical information.

### Medication features

The dataset contains 21 original medication-state variables in addition to engineered medication aggregate features. These variables require a separate redundancy and sparsity analysis before a final keep-or-drop decision is made.

### Engineered binary indicators

Binary variables representing documentation status and treatment-related information will generally be treated as categorical or binary features.

### Engineered laboratory measurement features

The engineered glucose and A1C variables require special evaluation because the underlying measurements are available for only a small proportion of encounters. Their missingness may itself contain useful information and must not be handled through automatic imputation without further analysis.

This classification is an initial framework and does not yet represent the final feature set. The next steps will systematically investigate redundancy, sparsity, cardinality, and potential information leakage.


In [7]:
# ============================================================
# STEP 1.2 — INITIAL FEATURE CLASSIFICATION
# ============================================================

# Identifiers
identifier_features = [
    "encounter_id",
    "patient_nbr"
]

# Target-related features
target_features = [
    "readmitted",
    "readmitted_30d"
]

# Modeling population / eligibility features
eligibility_features = [
    "death_related_disposition"
]

# Numerical category IDs replaced by descriptive mappings
mapped_id_features = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

# Corresponding descriptive mapped features
mapped_categorical_features = [
    "admission_type",
    "discharge_disposition",
    "admission_source"
]

# Medication state features
medication_features = [
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone"
]

# Medication aggregate features
medication_aggregate_features = [
    "num_medications_active",
    "num_medications_up",
    "num_medications_down",
    "num_medications_steady"
]

# Engineered binary features
binary_features = [
    "medication_changed",
    "diabetes_medication_used",
    "weight_documented",
    "max_glu_serum_documented",
    "A1C_documented"
]

# Diagnosis group features
diagnosis_group_features = [
    "diag_1_group",
    "diag_2_group",
    "diag_3_group"
]

# Engineered laboratory measurement features
engineered_lab_features = [
    "max_glu_serum_level",
    "A1C_level"
]

# Genuine numeric utilization / count features
numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

# Assign feature category
def classify_feature(feature):
    
    if feature in identifier_features:
        return "Identifier"
    
    elif feature in target_features:
        return "Target-related"
    
    elif feature in eligibility_features:
        return "Eligibility / population construction"
    
    elif feature in mapped_id_features:
        return "Mapped ID (redundant version)"
    
    elif feature in mapped_categorical_features:
        return "Categorical - mapped clinical / operational"
    
    elif feature in medication_features:
        return "Categorical - medication state"
    
    elif feature in medication_aggregate_features:
        return "Numeric - medication aggregate"
    
    elif feature in binary_features:
        return "Binary engineered"
    
    elif feature in diagnosis_group_features:
        return "Categorical - diagnosis group"
    
    elif feature in engineered_lab_features:
        return "Engineered laboratory measurement"
    
    elif feature in numeric_features:
        return "Numeric - utilization / count"
    
    else:
        return "Categorical / requires review"


feature_audit["initial_category"] = feature_audit["feature"].apply(
    classify_feature
)

# Display classification summary
feature_classification_summary = (
    feature_audit["initial_category"]
    .value_counts()
    .rename_axis("initial_category")
    .reset_index(name="feature_count")
)

print("FEATURE CLASSIFICATION SUMMARY")
print("=" * 60)

display(feature_classification_summary)

print("\nDETAILED FEATURE AUDIT")
print("=" * 60)

display(feature_audit)

FEATURE CLASSIFICATION SUMMARY


,initial_category,feature_count
0,Categorical - medication state,21
1,Categorical / requires review,13
2,Numeric - utilization / count,8
3,Binary engineered,5
4,Numeric - medication aggregate,4
5,Categorical - mapped clinical / operational,3
6,Mapped ID (redundant version),3
7,Categorical - diagnosis group,3
8,Engineered laboratory measurement,2
9,Identifier,2



DETAILED FEATURE AUDIT


,feature,data_type,missing_count,missing_pct,unique_count,unique_pct,sample_values,initial_category
0,A1C_documented,int64,0,0.00,2,0.00,"[0, 1]",Binary engineered
1,A1C_level,float64,83238,83.14,3,0.00,"[1.0, 2.0, 0.0]",Engineered laboratory measurement
2,A1Cresult,str,0,0.00,4,0.00,"[Not_Documented, >7, >8, Norm]",Categorical / requires review
3,acarbose,str,0,0.00,4,0.00,"[No, Steady, Up, Down]",Categorical - medication state
4,acetohexamide,str,0,0.00,2,0.00,"[No, Steady]",Categorical - medication state
...,...,...,...,...,...,...,...,...
62,tolazamide,str,0,0.00,3,0.00,"[No, Steady, Up]",Categorical - medication state
63,tolbutamide,str,0,0.00,2,0.00,"[No, Steady]",Categorical - medication state
64,troglitazone,str,0,0.00,2,0.00,"[No, Steady]",Categorical - medication state
65,weight,str,96958,96.85,9,0.01,"[[75-100), [50-75), [0-25), [100-125), [25-50)]",Categorical / requires review


## STEP 1.3 — Definition and Validation of Provisional Modeling Features

Before conducting detailed feature analysis, the provisional modeling feature set must be explicitly defined and validated.

The modeling dataset contains 67 columns. Eight columns are excluded from the candidate feature set for specific methodological reasons.

### Identifier columns

`encounter_id` and `patient_nbr` are identifiers rather than direct predictive variables.

The encounter identifier is unique to each hospital encounter and does not contain meaningful clinical information for prediction.

The patient identifier is also excluded from the direct feature set. However, it will be retained separately because patient-level information may be required when designing the train, validation, and test splitting strategy.

### Target-related columns

`readmitted` is the original outcome column used to construct the binary target.

`readmitted_30d` is the final machine learning target.

Neither variable can be included as a predictor because doing so would directly reveal outcome information.

### Modeling population construction feature

`death_related_disposition` was used to identify and remove death-related encounters when constructing the final modeling population.

Because this variable defines eligibility for the prediction population rather than an independent predictor for the intended modeling problem, it is excluded from the candidate feature set.

### Redundant numerical category identifiers

The numerical variables `admission_type_id`, `discharge_disposition_id`, and `admission_source_id` have already been mapped to descriptive categorical versions.

The descriptive variables are initially retained because they represent the same underlying information while providing improved interpretability.

After excluding these eight variables, the provisional modeling feature set contains:

**59 candidate features**

The remaining feature audit will focus primarily on these 59 candidate predictors.


In [9]:
# ============================================================
# STEP 1.3 — PROVISIONAL MODEL FEATURE VALIDATION
# ============================================================

# Define excluded columns
excluded_columns = [
    "encounter_id",
    "patient_nbr",
    "readmitted",
    "readmitted_30d",
    "death_related_disposition",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

# Define target
target_column = "readmitted_30d"

# Create provisional model feature list
provisional_features = [
    column
    for column in df_model_feature.columns
    if column not in excluded_columns
]

# ============================================================
# VALIDATION
# ============================================================

print("PROVISIONAL FEATURE SET VALIDATION")
print("=" * 60)

print(f"Total columns in modeling dataset: {df_model_feature.shape[1]}")
print(f"Excluded columns: {len(excluded_columns)}")
print(f"Provisional model features: {len(provisional_features)}")

expected_feature_count = 59

if len(provisional_features) == expected_feature_count:
    print("\nPASS — Correct number of provisional model features")
else:
    print("\nFAIL — Unexpected provisional feature count")

# Check that excluded columns exist
missing_excluded_columns = [
    column
    for column in excluded_columns
    if column not in df_model_feature.columns
]

if len(missing_excluded_columns) == 0:
    print("PASS — All excluded columns are present in the dataset")
else:
    print(
        "FAIL — Missing excluded columns:",
        missing_excluded_columns
    )

# Check that the target is not included as a feature
if target_column not in provisional_features:
    print("PASS — Target is not included in provisional features")
else:
    print("FAIL — Target leakage detected")

# Create feature-only dataset
X_provisional = df_model_feature[provisional_features].copy()

print("\nFeature Matrix Shape")
print(X_provisional.shape)

print("\nProvisional Features:")
print(provisional_features)

PROVISIONAL FEATURE SET VALIDATION
Total columns in modeling dataset: 67
Excluded columns: 8
Provisional model features: 59

PASS — Correct number of provisional model features
PASS — All excluded columns are present in the dataset
PASS — Target is not included in provisional features

Feature Matrix Shape
(100114, 59)

Provisional Features:
['race', 'gender', 'age', 'weight', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'chang

## STEP 1.4 — Missingness, Sparsity, and Dominant Category Analysis

The next stage of the feature audit examines the distribution of information within the 59 provisional model features.

Missingness and sparsity are not the same concept and should be evaluated separately.

### Missingness

A feature contains missing information when one or more observations contain null values.

In the current dataset, the variables `weight`, `max_glu_serum_level`, and `A1C_level` contain substantial missingness.

These variables require special attention because their missingness is related to clinical documentation. Documentation indicator variables have already been engineered for each measurement. Therefore, the missing values should not be automatically imputed without considering the relationship between measurement availability and the engineered documentation indicators.

### Structural sparsity

A feature may contain no missing values while still providing very little variation.

This is particularly relevant for medication-state variables. A medication column may contain a valid value for every encounter while the overwhelming majority of observations belong to the `No` category.

Such features are structurally sparse because the medication was not actively used or changed for most encounters.

### Dominant categories

For categorical variables, the proportion represented by the most frequent category provides an indication of feature imbalance.

A feature with an extremely dominant category may have limited predictive information. However, dominance alone is not sufficient evidence for automatic removal because rare clinical events or treatment patterns may still provide useful predictive signal.

### Objective of this analysis

For every provisional model feature, the analysis will calculate:

* missing value count;
* missing value percentage;
* number of unique values;
* most frequent value;
* percentage represented by the most frequent value;
* percentage represented by all non-dominant values combined.

This analysis will identify candidate features for further investigation, particularly:

* features with substantial missingness;
* highly dominant categorical features;
* near-constant variables;
* sparse medication variables;
* variables requiring specialized preprocessing.

No feature will be automatically removed during this stage solely because it is sparse or highly imbalanced.


In [10]:
# ============================================================
# STEP 1.4 — MISSINGNESS AND SPARSITY AUDIT
# ============================================================

sparsity_results = []

for column in provisional_features:
    
    series = X_provisional[column]
    
    # Missing values
    missing_count = series.isna().sum()
    missing_pct = round(series.isna().mean() * 100, 2)
    
    # Unique non-missing values
    unique_count = series.nunique(dropna=True)
    
    # Most frequent non-missing value
    non_missing_series = series.dropna()
    
    if len(non_missing_series) > 0:
        
        value_distribution = non_missing_series.value_counts(
            dropna=False
        )
        
        dominant_value = value_distribution.index[0]
        
        dominant_count = value_distribution.iloc[0]
        
        dominant_pct = round(
            dominant_count / len(series) * 100,
            2
        )
        
        non_dominant_pct = round(
            100 - dominant_pct - missing_pct,
            2
        )
    
    else:
        dominant_value = None
        dominant_count = 0
        dominant_pct = 0
        non_dominant_pct = 0
    
    sparsity_results.append({
        "feature": column,
        "data_type": str(series.dtype),
        "missing_count": missing_count,
        "missing_pct": missing_pct,
        "unique_count": unique_count,
        "dominant_value": dominant_value,
        "dominant_count": dominant_count,
        "dominant_pct": dominant_pct,
        "non_dominant_pct": non_dominant_pct
    })


sparsity_audit = pd.DataFrame(sparsity_results)

# Sort by missingness first, then dominant category percentage
sparsity_audit = (
    sparsity_audit
    .sort_values(
        by=["missing_pct", "dominant_pct"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

print("MISSINGNESS AND SPARSITY AUDIT")
print("=" * 80)

display(sparsity_audit)

MISSINGNESS AND SPARSITY AUDIT


,feature,data_type,missing_count,missing_pct,unique_count,dominant_value,dominant_count,dominant_pct,non_dominant_pct
0,weight,str,96958,96.85,9,[75-100),1320,1.32,1.83
1,max_glu_serum_level,float64,94890,94.78,3,0.0,2573,2.57,2.65
2,A1C_level,float64,83238,83.14,3,2.0,8151,8.14,8.72
3,acetohexamide,str,0,0.00,2,No,100113,100.00,0.00
4,troglitazone,str,0,0.00,2,No,100111,100.00,0.00
5,glimepiride-pioglitazone,str,0,0.00,2,No,100113,100.00,0.00
6,metformin-rosiglitazone,str,0,0.00,2,No,100112,100.00,0.00
7,metformin-pioglitazone,str,0,0.00,2,No,100113,100.00,0.00
8,glipizide-metformin,str,0,0.00,2,No,100101,99.99,0.01
9,tolbutamide,str,0,0.00,2,No,100093,99.98,0.02


In [11]:
# ============================================================
# FEATURES WITH HIGH MISSINGNESS
# ============================================================

high_missing_features = sparsity_audit[
    sparsity_audit["missing_pct"] > 0
].sort_values(
    by="missing_pct",
    ascending=False
)

print("FEATURES WITH MISSING VALUES")
print("=" * 60)

display(high_missing_features)


# ============================================================
# FEATURES WITH EXTREMELY DOMINANT VALUES
# ============================================================

high_dominance_features = sparsity_audit[
    sparsity_audit["dominant_pct"] >= 95
].sort_values(
    by="dominant_pct",
    ascending=False
)

print("\nFEATURES WITH DOMINANT VALUE >= 95%")
print("=" * 60)

display(high_dominance_features)


# ============================================================
# FEATURES WITH EXTREMELY LOW VARIATION
# ============================================================

near_constant_features = sparsity_audit[
    sparsity_audit["dominant_pct"] >= 99
].sort_values(
    by="dominant_pct",
    ascending=False
)

print("\nFEATURES WITH DOMINANT VALUE >= 99%")
print("=" * 60)

display(near_constant_features)

FEATURES WITH MISSING VALUES


,feature,data_type,missing_count,missing_pct,unique_count,dominant_value,dominant_count,dominant_pct,non_dominant_pct
0,weight,str,96958,96.85,9,[75-100),1320,1.32,1.83
1,max_glu_serum_level,float64,94890,94.78,3,0.0,2573,2.57,2.65
2,A1C_level,float64,83238,83.14,3,2.0,8151,8.14,8.72



FEATURES WITH DOMINANT VALUE >= 95%


,feature,data_type,missing_count,missing_pct,unique_count,dominant_value,dominant_count,dominant_pct,non_dominant_pct
3,acetohexamide,str,0,0.0,2,No,100113,100.00,0.00
4,troglitazone,str,0,0.0,2,No,100111,100.00,0.00
5,glimepiride-pioglitazone,str,0,0.0,2,No,100113,100.00,0.00
6,metformin-rosiglitazone,str,0,0.0,2,No,100112,100.00,0.00
7,metformin-pioglitazone,str,0,0.0,2,No,100113,100.00,0.00
8,glipizide-metformin,str,0,0.0,2,No,100101,99.99,0.01
9,tolbutamide,str,0,0.0,2,No,100093,99.98,0.02
10,miglitol,str,0,0.0,4,No,100076,99.96,0.04
11,tolazamide,str,0,0.0,3,No,100075,99.96,0.04
12,chlorpropamide,str,0,0.0,4,No,100029,99.92,0.08



FEATURES WITH DOMINANT VALUE >= 99%


,feature,data_type,missing_count,missing_pct,unique_count,dominant_value,dominant_count,dominant_pct,non_dominant_pct
3,acetohexamide,str,0,0.0,2,No,100113,100.00,0.00
4,troglitazone,str,0,0.0,2,No,100111,100.00,0.00
5,glimepiride-pioglitazone,str,0,0.0,2,No,100113,100.00,0.00
6,metformin-rosiglitazone,str,0,0.0,2,No,100112,100.00,0.00
7,metformin-pioglitazone,str,0,0.0,2,No,100113,100.00,0.00
8,glipizide-metformin,str,0,0.0,2,No,100101,99.99,0.01
9,tolbutamide,str,0,0.0,2,No,100093,99.98,0.02
10,miglitol,str,0,0.0,4,No,100076,99.96,0.04
11,tolazamide,str,0,0.0,3,No,100075,99.96,0.04
12,chlorpropamide,str,0,0.0,4,No,100029,99.92,0.08


## STEP 1.5 — Medication Feature Activity and Sparsity Analysis

The initial sparsity audit identified substantial variation in the prevalence of the 21 original medication-state features.

A medication feature can contain no missing values while still being structurally sparse because most encounters have the `No` category.

To evaluate the medication variables more precisely, the analysis will calculate the number and percentage of encounters in which each medication has a state other than `No`.

The medication-state variables can therefore be evaluated based on:

* number of encounters with active or changed medication status;
* percentage of encounters with a non-`No` medication state;
* number of observed medication states;
* detailed distribution across `No`, `Steady`, `Up`, and `Down`.

The objective is to distinguish between:

1. medication variables with almost no variation;
2. extremely rare medication patterns;
3. sparse but potentially meaningful medication variables; and
4. commonly observed medication features.

No medication feature will be removed solely based on the initial dominant-category analysis. Final decisions will also consider redundancy with the engineered medication aggregate features and the potential clinical information represented by individual medications.


In [12]:
# ============================================================
# STEP 1.5 — MEDICATION FEATURE ACTIVITY ANALYSIS
# ============================================================

medication_activity_results = []

for medication in medication_features:
    
    medication_series = X_provisional[medication]
    
    total_count = len(medication_series)
    
    no_count = (medication_series == "No").sum()
    
    active_count = (medication_series != "No").sum()
    
    active_pct = round(
        active_count / total_count * 100,
        4
    )
    
    unique_states = medication_series.nunique()
    
    medication_activity_results.append({
        "medication": medication,
        "total_encounters": total_count,
        "no_count": no_count,
        "active_count": active_count,
        "active_pct": active_pct,
        "unique_states": unique_states
    })


medication_activity_audit = pd.DataFrame(
    medication_activity_results
)

medication_activity_audit = (
    medication_activity_audit
    .sort_values(
        by="active_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

print("MEDICATION ACTIVITY SUMMARY")
print("=" * 70)

display(medication_activity_audit)

MEDICATION ACTIVITY SUMMARY


,medication,total_encounters,no_count,active_count,active_pct,unique_states
0,insulin,100114,46680,53434,53.3732,4
1,metformin,100114,80216,19898,19.8753,4
2,glipizide,100114,87509,12605,12.5906,4
3,glyburide,100114,89547,10567,10.5550,4
4,pioglitazone,100114,92833,7281,7.2727,4
5,rosiglitazone,100114,93785,6329,6.3218,4
6,glimepiride,100114,94967,5147,5.1411,4
7,repaglinide,100114,98587,1527,1.5253,4
8,glyburide-metformin,100114,99416,698,0.6972,4
9,nateglinide,100114,99419,695,0.6942,4


In [13]:
# ============================================================
# STEP 1.5 — MEDICATION FEATURE ACTIVITY ANALYSIS
# ============================================================

medication_activity_results = []

for medication in medication_features:
    
    medication_series = X_provisional[medication]
    
    total_count = len(medication_series)
    
    no_count = (medication_series == "No").sum()
    
    active_count = (medication_series != "No").sum()
    
    active_pct = round(
        active_count / total_count * 100,
        4
    )
    
    unique_states = medication_series.nunique()
    
    medication_activity_results.append({
        "medication": medication,
        "total_encounters": total_count,
        "no_count": no_count,
        "active_count": active_count,
        "active_pct": active_pct,
        "unique_states": unique_states
    })


medication_activity_audit = pd.DataFrame(
    medication_activity_results
)

medication_activity_audit = (
    medication_activity_audit
    .sort_values(
        by="active_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

print("MEDICATION ACTIVITY SUMMARY")
print("=" * 70)

display(medication_activity_audit)

MEDICATION ACTIVITY SUMMARY


,medication,total_encounters,no_count,active_count,active_pct,unique_states
0,insulin,100114,46680,53434,53.3732,4
1,metformin,100114,80216,19898,19.8753,4
2,glipizide,100114,87509,12605,12.5906,4
3,glyburide,100114,89547,10567,10.5550,4
4,pioglitazone,100114,92833,7281,7.2727,4
5,rosiglitazone,100114,93785,6329,6.3218,4
6,glimepiride,100114,94967,5147,5.1411,4
7,repaglinide,100114,98587,1527,1.5253,4
8,glyburide-metformin,100114,99416,698,0.6972,4
9,nateglinide,100114,99419,695,0.6942,4


In [14]:
# ============================================================
# DETAILED MEDICATION STATE DISTRIBUTION
# ============================================================

medication_state_distribution = pd.DataFrame({
    medication: (
        X_provisional[medication]
        .value_counts(normalize=True)
        .mul(100)
        .round(4)
    )
    for medication in medication_features
}).T

medication_state_distribution.index.name = "medication"

print("MEDICATION STATE DISTRIBUTION (%)")
print("=" * 70)

display(medication_state_distribution)

MEDICATION STATE DISTRIBUTION (%)


,Down,No,Steady,Up
medication,,,,
metformin,0.5743,80.1247,18.2352,1.0658
repaglinide,0.0449,98.4747,1.3714,0.1089
nateglinide,0.0110,99.3058,0.6592,0.0240
chlorpropamide,0.0010,99.9151,0.0779,0.0060
glimepiride,0.1928,94.8589,4.6247,0.3236
acetohexamide,NaN,99.9990,0.0010,NaN
glipizide,0.5534,87.4094,11.2712,0.7661
glyburide,0.5604,89.4450,9.1895,0.8051
tolbutamide,NaN,99.9790,0.0210,NaN


## 4. Medication Feature Sparsity Analysis

The dataset contains 21 individual diabetes medication features. Each medication describes the medication state during the hospital encounter using categories such as:

- `No`
- `Steady`
- `Up`
- `Down`

Before including all medication variables in the machine learning model, feature sparsity was evaluated.

### Why this analysis is important

Some medications are extremely rare in the dataset. Features with almost all observations in a single category can:

- contribute very little predictive information,
- create unnecessary model complexity,
- introduce unstable estimates for rare categories,
- increase dimensionality after one-hot encoding,
- potentially cause overfitting.

However, sparsity alone is not sufficient reason to remove a feature. A rare medication may still contain clinically meaningful predictive information.

Therefore, the medication features will be evaluated using:

1. Medication prevalence
2. Number of active encounters
3. Distribution across medication states
4. Redundancy with engineered medication aggregate features
5. Predictive contribution during later model comparison

### Initial Findings

The medication analysis shows substantial differences in prevalence.

#### Common medications

The most frequently active medications are:

- Insulin
- Metformin
- Glipizide
- Glyburide
- Pioglitazone
- Rosiglitazone
- Glimepiride

These features have sufficient variation and should initially remain candidates for modeling.

#### Moderately rare medications

Some medications are active in a relatively small proportion of encounters, including:

- Repaglinide
- Glyburide-metformin
- Nateglinide
- Acarbose

These features should not automatically be removed. Their predictive contribution will be evaluated further.

#### Extremely sparse medications

Several medication features are active in fewer than approximately 0.1% of encounters. Some occur in only a handful of observations.

Examples include:

- Acetohexamide
- Troglitazone
- Glimepiride-pioglitazone
- Metformin-rosiglitazone
- Metformin-pioglitazone
- Glipizide-metformin
- Tolbutamide
- Miglitol
- Tolazamide
- Chlorpropamide

These variables provide very limited variation.

At this stage, they are flagged for removal from the primary modeling feature set, subject to final feature strategy validation.

### Important Modeling Decision

The project contains two types of medication information:

#### Individual medication features

These describe the state of specific medications.

#### Engineered medication aggregate features

These summarize medication activity across all diabetes medications:

- `num_medications_active`
- `num_medications_up`
- `num_medications_down`
- `num_medications_steady`
- `medication_changed`
- `diabetes_medication_used`

The aggregate features provide a compressed representation of overall medication treatment intensity and changes.

Because the individual medication variables contain drug-specific information while the aggregate features contain overall treatment information, they are not automatically considered duplicates.

The final modeling strategy will therefore compare the value of individual medication features against the engineered aggregate features rather than blindly removing one group.

### 4.1 Identification of Individual Medication Features

The dataset contains individual medication variables that describe the status of diabetes-related medications during each hospital encounter.

The medication states are generally represented as:

- `No`
- `Steady`
- `Up`
- `Down`

For this analysis, a medication is considered **active** when its value is not equal to `No`.

The following analysis calculates the prevalence of each medication and identifies features with extremely limited variation.

In [16]:
# ============================================================
# IDENTIFY INDIVIDUAL MEDICATION FEATURES
# ============================================================

medication_columns = [
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone"
]

# Validation
missing_medication_columns = [
    col for col in medication_columns
    if col not in df_model_feature.columns
]

print("=" * 65)
print("MEDICATION FEATURE VALIDATION")
print("=" * 65)

print(f"\nExpected medication features: {len(medication_columns)}")
print(f"Medication features found: "
      f"{len(medication_columns) - len(missing_medication_columns)}")

if len(missing_medication_columns) == 0:
    print("\nPASS — All medication columns are present")
else:
    print("\nWARNING — Missing medication columns:")
    print(missing_medication_columns)

MEDICATION FEATURE VALIDATION

Expected medication features: 21
Medication features found: 21

PASS — All medication columns are present


### 4.2 Medication Prevalence Analysis

To evaluate the usefulness of individual medication features, medication prevalence was calculated.

For each medication, the analysis includes:

- Total number of encounters
- Number of encounters where the medication was not used
- Number of encounters where the medication was active
- Percentage of encounters where the medication was active
- Number of observed medication states

A medication is considered active when its recorded state is:

- `Steady`
- `Up`
- `Down`

The value `No` indicates that the medication was not active during the encounter.

In [17]:
# ============================================================
# CREATE MEDICATION PREVALENCE SUMMARY
# ============================================================

medication_summary_list = []

for medication in medication_columns:
    
    total_encounters = len(df_model_feature)
    
    no_count = (
        df_model_feature[medication]
        .eq("No")
        .sum()
    )
    
    active_count = total_encounters - no_count
    
    active_pct = (
        active_count / total_encounters
    ) * 100
    
    unique_states = (
        df_model_feature[medication]
        .nunique()
    )
    
    medication_summary_list.append({
        "medication": medication,
        "total_encounters": total_encounters,
        "no_count": no_count,
        "active_count": active_count,
        "active_pct": round(active_pct, 4),
        "unique_states": unique_states
    })


medication_summary = pd.DataFrame(
    medication_summary_list
)

medication_summary = (
    medication_summary
    .sort_values(
        by="active_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

display(medication_summary)

,medication,total_encounters,no_count,active_count,active_pct,unique_states
0,insulin,100114,46680,53434,53.3732,4
1,metformin,100114,80216,19898,19.8753,4
2,glipizide,100114,87509,12605,12.5906,4
3,glyburide,100114,89547,10567,10.5550,4
4,pioglitazone,100114,92833,7281,7.2727,4
5,rosiglitazone,100114,93785,6329,6.3218,4
6,glimepiride,100114,94967,5147,5.1411,4
7,repaglinide,100114,98587,1527,1.5253,4
8,glyburide-metformin,100114,99416,698,0.6972,4
9,nateglinide,100114,99419,695,0.6942,4


In [18]:
# ============================================================
# VALIDATE MEDICATION SUMMARY
# ============================================================

print("=" * 65)
print("MEDICATION SUMMARY VALIDATION")
print("=" * 65)

# Check 1: Expected number of medications
expected_medication_count = 21
actual_medication_count = len(medication_summary)

if actual_medication_count == expected_medication_count:
    print("\nPASS — Correct number of medication features")
else:
    print(
        f"\nFAIL — Expected {expected_medication_count}, "
        f"found {actual_medication_count}"
    )

# Check 2: Active + No count should equal total encounters
count_check = (
    medication_summary["no_count"]
    + medication_summary["active_count"]
    == medication_summary["total_encounters"]
)

if count_check.all():
    print("PASS — Medication counts reconcile correctly")
else:
    print("FAIL — Medication count reconciliation failed")

# Check 3: Active percentage should be between 0 and 100
pct_check = (
    medication_summary["active_pct"]
    .between(0, 100)
)

if pct_check.all():
    print("PASS — All active percentages are valid")
else:
    print("FAIL — Invalid active percentage detected")

print("\nMedication summary shape:")
print(medication_summary.shape)

MEDICATION SUMMARY VALIDATION

PASS — Correct number of medication features
PASS — Medication counts reconcile correctly
PASS — All active percentages are valid

Medication summary shape:
(21, 6)


### 4.3 Identification of Extremely Sparse Medication Features

Medication features with very low active prevalence provide limited statistical variation.

Extremely sparse variables can create unstable category estimates and may contribute limited generalizable predictive value.

For the initial feature selection analysis, medications with an active prevalence below **0.1%** are flagged as extremely sparse.

This threshold is used as a feature review criterion rather than an automatic deletion rule.

In [19]:
# ============================================================
# SYSTEMATIC MEDICATION SPARSITY ANALYSIS
# ============================================================

SPARSITY_THRESHOLD_PCT = 0.1

extremely_sparse_medications = (
    medication_summary
    .loc[
        medication_summary["active_pct"] < SPARSITY_THRESHOLD_PCT,
        [
            "medication",
            "active_count",
            "active_pct",
            "unique_states"
        ]
    ]
    .sort_values("active_pct")
    .reset_index(drop=True)
)

print("=" * 65)
print("EXTREMELY SPARSE MEDICATION FEATURES")
print("=" * 65)

print(
    f"\nSparsity threshold: "
    f"active prevalence < {SPARSITY_THRESHOLD_PCT}%"
)

print(
    f"\nNumber of medication features flagged: "
    f"{len(extremely_sparse_medications)}"
)

display(extremely_sparse_medications)

EXTREMELY SPARSE MEDICATION FEATURES

Sparsity threshold: active prevalence < 0.1%

Number of medication features flagged: 10


,medication,active_count,active_pct,unique_states
0,acetohexamide,1,0.0010,2
1,glimepiride-pioglitazone,1,0.0010,2
2,metformin-pioglitazone,1,0.0010,2
3,metformin-rosiglitazone,2,0.0020,2
4,troglitazone,3,0.0030,2
5,glipizide-metformin,13,0.0130,2
6,tolbutamide,21,0.0210,2
7,miglitol,38,0.0380,4
8,tolazamide,39,0.0390,3
9,chlorpropamide,85,0.0849,4


### 4.4 Feature Selection Decision for Extremely Sparse Medications

The medication prevalence analysis identified 10 individual medication features with active prevalence below 0.1% of all encounters.

These medications were active in between 1 and 85 encounters out of 100,114 total encounters.

The affected features are:

- `acetohexamide`
- `glimepiride-pioglitazone`
- `metformin-pioglitazone`
- `metformin-rosiglitazone`
- `troglitazone`
- `glipizide-metformin`
- `tolbutamide`
- `miglitol`
- `tolazamide`
- `chlorpropamide`

These variables exhibit extremely limited variation. Including them in the primary model would create sparse categories after encoding and may produce unstable estimates or overfitting.

Therefore, these features will be excluded from the primary modeling feature set.

This decision is based on statistical representation in the dataset rather than clinical importance. A medication may be clinically important while still being unsuitable for reliable predictive modeling when it appears in too few observations.

The exclusion threshold is retained as a documented and reproducible feature-selection criterion:

> Extremely sparse medication feature: active prevalence < 0.1%

The remaining medication features will continue to be evaluated because they contain sufficient variation for further analysis.

In [21]:
# ============================================================
# FINAL EXTREMELY SPARSE MEDICATION EXCLUSION LIST
# ============================================================

sparse_medication_exclusions = (
    extremely_sparse_medications["medication"]
    .tolist()
)

print("=" * 65)
print("EXTREMELY SPARSE MEDICATION EXCLUSIONS")
print("=" * 65)

for i, feature in enumerate(sparse_medication_exclusions, start=1):
    print(f"{i}. {feature}")

print(f"\nTotal excluded medication features: "
      f"{len(sparse_medication_exclusions)}")

EXTREMELY SPARSE MEDICATION EXCLUSIONS
1. acetohexamide
2. glimepiride-pioglitazone
3. metformin-pioglitazone
4. metformin-rosiglitazone
5. troglitazone
6. glipizide-metformin
7. tolbutamide
8. miglitol
9. tolazamide
10. chlorpropamide

Total excluded medication features: 10


In [22]:
# ============================================================
# VALIDATE SPARSE MEDICATION EXCLUSIONS
# ============================================================

print("=" * 65)
print("SPARSE MEDICATION EXCLUSION VALIDATION")
print("=" * 65)

expected_sparse_features = 10

# Check 1
if len(sparse_medication_exclusions) == expected_sparse_features:
    print(
        f"\nPASS — Correct number of sparse medication "
        f"features identified: {expected_sparse_features}"
    )
else:
    print(
        f"\nFAIL — Expected {expected_sparse_features}, "
        f"found {len(sparse_medication_exclusions)}"
    )

# Check 2
missing_from_feature_set = [
    feature
    for feature in sparse_medication_exclusions
    if feature not in provisional_features
]

if not missing_from_feature_set:
    print(
        "PASS — All sparse medication exclusions "
        "exist in provisional features"
    )
else:
    print(
        "FAIL — Some sparse medication features are "
        "missing from provisional features:"
    )
    print(missing_from_feature_set)

# Check 3
print(
    f"\nProvisional feature count before exclusion: "
    f"{len(provisional_features)}"
)

print(
    f"Features removed due to extreme sparsity: "
    f"{len(sparse_medication_exclusions)}"
)

print(
    f"Remaining provisional features: "
    f"{len(provisional_features) - len(sparse_medication_exclusions)}"
)

SPARSE MEDICATION EXCLUSION VALIDATION

PASS — Correct number of sparse medication features identified: 10
PASS — All sparse medication exclusions exist in provisional features

Provisional feature count before exclusion: 59
Features removed due to extreme sparsity: 10
Remaining provisional features: 49


In [23]:
# ============================================================
# REDUNDANT FEATURE VALIDATION
# ============================================================

# Check whether engineered binary features are exact mappings
# of their original categorical variables.

redundancy_checks = pd.DataFrame({
    
    "feature_pair": [
        "change -> medication_changed",
        "diabetesMed -> diabetes_medication_used",
        "max_glu_serum -> max_glu_serum_documented",
        "A1Cresult -> A1C_documented",
        "weight -> weight_documented"
    ],
    
    "consistent": [
        (
            (df_model_feature["change"] == "Ch") ==
            (df_model_feature["medication_changed"] == 1)
        ).all(),
        
        (
            (df_model_feature["diabetesMed"] == "Yes") ==
            (df_model_feature["diabetes_medication_used"] == 1)
        ).all(),
        
        (
            (df_model_feature["max_glu_serum"] != "Not_Documented") ==
            (df_model_feature["max_glu_serum_documented"] == 1)
        ).all(),
        
        (
            (df_model_feature["A1Cresult"] != "Not_Documented") ==
            (df_model_feature["A1C_documented"] == 1)
        ).all(),
        
        (
            df_model_feature["weight"].notna() ==
            (df_model_feature["weight_documented"] == 1)
        ).all()
    ]
})

print("=" * 60)
print("REDUNDANT FEATURE VALIDATION")
print("=" * 60)

display(redundancy_checks)

REDUNDANT FEATURE VALIDATION


,feature_pair,consistent
0,change -> medication_changed,True
1,diabetesMed -> diabetes_medication_used,True
2,max_glu_serum -> max_glu_serum_documented,True
3,A1Cresult -> A1C_documented,True
4,weight -> weight_documented,True


In [24]:
# ============================================================
# CLINICAL FEATURE MAPPING VALIDATION
# ============================================================

print("=" * 70)
print("MAX GLUCOSE SERUM MAPPING")
print("=" * 70)

glu_mapping_check = (
    df_model_feature
    .groupby("max_glu_serum", dropna=False)["max_glu_serum_level"]
    .agg(
        unique_values=lambda x: sorted(x.dropna().unique().tolist()),
        non_null_count=lambda x: x.notna().sum(),
        missing_count=lambda x: x.isna().sum()
    )
    .reset_index()
)

display(glu_mapping_check)


print("=" * 70)
print("A1C RESULT MAPPING")
print("=" * 70)

a1c_mapping_check = (
    df_model_feature
    .groupby("A1Cresult", dropna=False)["A1C_level"]
    .agg(
        unique_values=lambda x: sorted(x.dropna().unique().tolist()),
        non_null_count=lambda x: x.notna().sum(),
        missing_count=lambda x: x.isna().sum()
    )
    .reset_index()
)

display(a1c_mapping_check)

MAX GLUCOSE SERUM MAPPING


,max_glu_serum,unique_values,non_null_count,missing_count
0,>200,[1.0],1440,0
1,>300,[2.0],1211,0
2,Norm,[0.0],2573,0
3,Not_Documented,[],0,94890


A1C RESULT MAPPING


,A1Cresult,unique_values,non_null_count,missing_count
0,>7,[1.0],3784,0
1,>8,[2.0],8151,0
2,Norm,[0.0],4941,0
3,Not_Documented,[],0,83238


In [25]:
# ============================================================
# PROVISIONAL FEATURE EXCLUSION LIST - VERSION 1
# ============================================================

features_to_drop_v1 = [
    
    # --------------------------------------------------------
    # Extremely rare medication features
    # --------------------------------------------------------
    "acetohexamide",
    "glimepiride-pioglitazone",
    "metformin-pioglitazone",
    "metformin-rosiglitazone",
    "troglitazone",
    "glipizide-metformin",
    "tolbutamide",
    "miglitol",
    "tolazamide",
    "chlorpropamide",
    
    # --------------------------------------------------------
    # Redundant engineered features
    # --------------------------------------------------------
    "medication_changed",
    "diabetes_medication_used",
    "max_glu_serum_documented",
    "A1C_documented",
    "max_glu_serum_level",
    "A1C_level"
]

print("Number of provisional features to drop:", len(features_to_drop_v1))
print("\nFeatures:")
for feature in features_to_drop_v1:
    print("-", feature)

Number of provisional features to drop: 16

Features:
- acetohexamide
- glimepiride-pioglitazone
- metformin-pioglitazone
- metformin-rosiglitazone
- troglitazone
- glipizide-metformin
- tolbutamide
- miglitol
- tolazamide
- chlorpropamide
- medication_changed
- diabetes_medication_used
- max_glu_serum_documented
- A1C_documented
- max_glu_serum_level
- A1C_level


In [27]:
# ============================================================
# FINAL PROVISIONAL FEATURE EXCLUSION LIST
# ============================================================

rare_medication_features = [
    "acetohexamide",
    "glimepiride-pioglitazone",
    "metformin-pioglitazone",
    "metformin-rosiglitazone",
    "troglitazone",
    "glipizide-metformin",
    "tolbutamide",
    "miglitol",
    "tolazamide",
    "chlorpropamide"
]

redundant_engineered_features = [
    "medication_changed",
    "diabetes_medication_used",
    "max_glu_serum_documented",
    "A1C_documented",
    "max_glu_serum_level",
    "A1C_level"
]
raw_diagnosis_features = [
    "diag_1",
    "diag_2",
    "diag_3"
]

weight_features_to_drop = [
    "weight"
]

features_to_drop = (

    # Extremely rare medication features
    rare_medication_features

    # Redundant engineered features
    + redundant_engineered_features

    # Raw high-cardinality diagnosis codes
    + raw_diagnosis_features

    # Extremely high-missingness weight category
    + weight_features_to_drop
)

print("=" * 70)
print("FEATURE SELECTION - PROVISIONAL EXCLUSIONS")
print("=" * 70)

print(f"\nTotal features to exclude: {len(features_to_drop)}")

for i, feature in enumerate(features_to_drop, start=1):
    print(f"{i}. {feature}")

FEATURE SELECTION - PROVISIONAL EXCLUSIONS

Total features to exclude: 20
1. acetohexamide
2. glimepiride-pioglitazone
3. metformin-pioglitazone
4. metformin-rosiglitazone
5. troglitazone
6. glipizide-metformin
7. tolbutamide
8. miglitol
9. tolazamide
10. chlorpropamide
11. medication_changed
12. diabetes_medication_used
13. max_glu_serum_documented
14. A1C_documented
15. max_glu_serum_level
16. A1C_level
17. diag_1
18. diag_2
19. diag_3
20. weight


In [56]:
# ============================================================
# CREATE PROVISIONAL FINAL FEATURE MATRIX
# ============================================================

selected_features = [
    feature
    for feature in provisional_features
    if feature not in features_to_drop
]

# We  have dorp  the Feature From orginal Dataset
# We  have  dropped those in x_selected
X_selected = df_model_feature[selected_features].copy()

print("=" * 70)
print("FINAL PROVISIONAL FEATURE SELECTION")
print("=" * 70)

print(f"\nOriginal provisional features: {len(provisional_features)}")
print(f"Excluded features: {len(features_to_drop)}")
print(f"Selected features: {len(selected_features)}")

print("\nSelected feature matrix shape:")
print(X_selected.shape)

print("\nSelected Features:")
for i, feature in enumerate(selected_features, start=1):
    print(f"{i}. {feature}")

FINAL PROVISIONAL FEATURE SELECTION

Original provisional features: 59
Excluded features: 20
Selected features: 39

Selected feature matrix shape:
(100114, 39)

Selected Features:
1. race
2. gender
3. age
4. time_in_hospital
5. payer_code
6. medical_specialty
7. num_lab_procedures
8. num_procedures
9. num_medications
10. number_outpatient
11. number_emergency
12. number_inpatient
13. number_diagnoses
14. max_glu_serum
15. A1Cresult
16. metformin
17. repaglinide
18. nateglinide
19. glimepiride
20. glipizide
21. glyburide
22. pioglitazone
23. rosiglitazone
24. acarbose
25. insulin
26. glyburide-metformin
27. change
28. diabetesMed
29. admission_type
30. discharge_disposition
31. admission_source
32. diag_1_group
33. diag_2_group
34. diag_3_group
35. num_medications_active
36. num_medications_up
37. num_medications_down
38. num_medications_steady
39. weight_documented


In [57]:
# ============================================================
# FINAL FEATURE SELECTION VALIDATION
# ============================================================

print("=" * 70)
print("FINAL FEATURE SELECTION VALIDATION")
print("=" * 70)

# 1. Count validation
print("\n1. FEATURE COUNT CHECK")

expected_selected_count = 39

if len(selected_features) == expected_selected_count:
    print(f"PASS — Selected feature count is correct: {len(selected_features)}")
else:
    print(
        f"FAIL — Expected {expected_selected_count}, "
        f"but found {len(selected_features)}"
    )


# 2. Duplicate feature validation
print("\n2. DUPLICATE FEATURE CHECK")

duplicate_selected = (
    pd.Series(selected_features)
    .value_counts()
)

duplicate_selected = duplicate_selected[
    duplicate_selected > 1
]

if duplicate_selected.empty:
    print("PASS — No duplicate features found")
else:
    print("FAIL — Duplicate features found:")
    display(duplicate_selected)


# 3. Selected features exist in dataset
print("\n3. FEATURE EXISTENCE CHECK")

missing_selected_features = [
    feature
    for feature in selected_features
    if feature not in df_model_feature.columns
]

if len(missing_selected_features) == 0:
    print("PASS — All selected features exist in the modeling dataset")
else:
    print("FAIL — Missing selected features:")
    print(missing_selected_features)


# 4. Excluded features are not selected
print("\n4. EXCLUSION CHECK")

incorrectly_retained = [
    feature
    for feature in features_to_drop
    if feature in selected_features
]

if len(incorrectly_retained) == 0:
    print("PASS — No excluded features remain in selected features")
else:
    print("FAIL — Some excluded features were retained:")
    print(incorrectly_retained)


# 5. Target leakage check
print("\n5. TARGET LEAKAGE CHECK")

target_related_features = [
    "readmitted",
    "readmitted_binary"
]

leakage_features = [
    feature
    for feature in target_related_features
    if feature in selected_features
]

if len(leakage_features) == 0:
    print("PASS — No target-related features found")
else:
    print("FAIL — Potential target leakage detected:")
    print(leakage_features)


# 6. Final matrix validation
print("\n6. FINAL MATRIX CHECK")

X_selected = df_model_feature[selected_features].copy()

print(f"Expected shape: ({len(df_model_feature)}, {len(selected_features)})")
print(f"Actual shape:   {X_selected.shape}")

if X_selected.shape == (
    len(df_model_feature),
    len(selected_features)
):
    print("PASS — Final feature matrix shape is correct")
else:
    print("FAIL — Feature matrix shape mismatch")


print("\n" + "=" * 70)
print("VALIDATION COMPLETE")
print("=" * 70)

FINAL FEATURE SELECTION VALIDATION

1. FEATURE COUNT CHECK
PASS — Selected feature count is correct: 39

2. DUPLICATE FEATURE CHECK
PASS — No duplicate features found

3. FEATURE EXISTENCE CHECK
PASS — All selected features exist in the modeling dataset

4. EXCLUSION CHECK
PASS — No excluded features remain in selected features

5. TARGET LEAKAGE CHECK
PASS — No target-related features found

6. FINAL MATRIX CHECK
Expected shape: (100114, 39)
Actual shape:   (100114, 39)
PASS — Final feature matrix shape is correct

VALIDATION COMPLETE


In [53]:
# ============================================================
# IDENTIFY TARGET-RELATED COLUMNS
# ============================================================

target_candidates = [
    column
    for column in df_model_feature.columns
    if any(
        keyword in column.lower()
        for keyword in [
            "readmit",
            "target",
            "outcome",
            "label"
        ]
    )
]

print("=" * 70)
print("TARGET-RELATED COLUMN SEARCH")
print("=" * 70)

print("\nPossible target columns:")

for column in target_candidates:
    print(f"\n{column}")
    print("Unique values:", df_model_feature[column].unique())
    print("Value counts:")
    print(df_model_feature[column].value_counts(dropna=False))

TARGET-RELATED COLUMN SEARCH

Possible target columns:

readmitted
Unique values: <StringArray>
['NO', '>30', '<30']
Length: 3, dtype: str
Value counts:
readmitted
NO     53212
>30    35545
<30    11357
Name: count, dtype: int64

readmitted_30d
Unique values: [0 1]
Value counts:
readmitted_30d
0    88757
1    11357
Name: count, dtype: int64


In [58]:
# ============================================================
# TARGET VARIABLE DEFINITION AND VALIDATION
# ============================================================

target_column = "readmitted_30d"

# Create target vector
y = df_model_feature[target_column].copy()

print("=" * 70)
print("TARGET VARIABLE VALIDATION")
print("=" * 70)

print(f"\nTarget column: {target_column}")
print(f"Target shape: {y.shape}")

print("\nTarget value counts:")
print(y.value_counts())

print("\nTarget distribution (%):")
print(y.value_counts(normalize=True).mul(100).round(2))

# Validation
assert target_column in df_model_feature.columns, \
    "ERROR — Target column not found"

assert set(y.unique()) == {0, 1}, \
    "ERROR — Target is not binary"

assert y.isna().sum() == 0, \
    "ERROR — Target contains missing values"

print("\nPASS — Target column exists")
print("PASS — Target is binary")
print("PASS — Target contains no missing values")

print("\nTarget definition:")
print("0 = No readmission within 30 days")
print("1 = Readmission within 30 days")

print("=" * 70)

TARGET VARIABLE VALIDATION

Target column: readmitted_30d
Target shape: (100114,)

Target value counts:
readmitted_30d
0    88757
1    11357
Name: count, dtype: int64

Target distribution (%):
readmitted_30d
0    88.66
1    11.34
Name: proportion, dtype: float64

PASS — Target column exists
PASS — Target is binary
PASS — Target contains no missing values

Target definition:
0 = No readmission within 30 days
1 = Readmission within 30 days


In [61]:
target_column

'readmitted_30d'

In [66]:
# ============================================================
# CREATE FINAL MACHINE LEARNING DATASET
# ============================================================



final_columns = selected_features + [target_column]
df_final_model = df_model_feature[final_columns].copy()

print("=" * 70)
print("FINAL MACHINE LEARNING DATASET")
print("=" * 70)

print(f"\nShape: {df_final_model.shape}")
print(f"Features: {len(selected_features)}")
print(f"Target: {target_column}")

print("\nTarget exists:")
print(target_column in df_final_model.columns)

FINAL MACHINE LEARNING DATASET

Shape: (100114, 40)
Features: 39
Target: readmitted_30d

Target exists:
True


In [69]:
df_model_feature.to_csv('../Data/Processed/Diabetes_orginal_file.csv', index=False)# orginal File with all col
df_final_model.to_csv("../Data/Processed/diabetes_model_ready.csv", index=False)

# Feature Selection and Target Definition

## Feature Selection

The feature selection process was completed after evaluating the provisional modeling features for:

- Missing data patterns
- Near-constant features
- Extremely sparse medication variables
- Feature redundancy
- Engineered feature consistency
- Potential target leakage

The modeling dataset initially contained **59 provisional features**.

After removing redundant, extremely sparse, and near-constant features, the final feature set contains **39 features**.

## Final Feature Set

The final selected features include:

- Patient demographic information
- Hospital utilization metrics
- Clinical and diagnostic information
- Laboratory result categories
- Relevant diabetes medications
- Medication activity and change indicators
- Admission and discharge information

All selected features were validated for:

- Correct feature count
- Duplicate features
- Feature existence
- Excluded feature removal
- Target leakage
- Final feature matrix shape

**Final feature matrix shape:**

```text
(100114, 39)